# Análisis de Rotación - Topic Modeling

## Objetivo
Analizar las respuestas de encuestas sobre rotación mediante técnicas de topic modeling (LDA) para identificar patrones y temas recurrentes en las percepciones de los gerentes.

## Contenido
1. Configuración del entorno
2. Carga de datos
3. Procesamiento y limpieza
4. Integración de datos
5. Análisis de topics
6. Funciones de utilidad

## Preguntas de la Encuesta
1. ¿Qué crees que ha hecho que algunas personas lleven tanto tiempo en la empresa?
2. ¿A qué atribuyes que algunos Colaboradores renuncien teniendo poco tiempo laborando en la empresa?
3. ¿A qué atribuyes que algunos Colaboradores renuncien teniendo mucho tiempo laborando en la empresa?
4. ¿Qué condiciones internas y/o del entorno (de la localidad) hacen que liderar este centro sea más retador que otros?
5. ¿Qué apoyos, decisiones o herramientas (internas o externas) podrían ayudar a mejorar la permanencia?
6. ¿Hay algo que no te hayamos preguntado y que consideres importante agregar para poder comprender por qué los Colaboradores toman la decisión de salir de la empresa?

## 1. Configuración del Entorno

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from datetime import datetime, date, time, timedelta
from dateutil.relativedelta import relativedelta

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

## 2. Carga de Datos

### 2.1. Topics de LDA de Respuestas de Encuestas

In [15]:
lda_topics_df = pd.read_csv("../data/LDA_res.csv")
lda_topics_df.drop(columns=["Unnamed: 0"], inplace=True)
print(f"Dimensiones del dataset de topics: {lda_topics_df.shape}")
lda_topics_df.head()

Dimensiones del dataset de topics: (5033, 120)


,P1_n_0,P1_n_1,P1_n_2,P1_n_3,P1_n_4,P1_n_5,P1_n_6,P1_n_7,P1_n_8,P1_n_9,P1_v_0,P1_v_1,P1_v_2,P1_v_3,P1_v_4,P1_v_5,P1_v_6,P1_v_7,P1_v_8,P1_v_9,P2_n_0,P2_n_1,P2_n_2,P2_n_3,P2_n_4,P2_n_5,P2_n_6,P2_n_7,P2_n_8,P2_n_9,P2_v_0,P2_v_1,P2_v_2,P2_v_3,P2_v_4,P2_v_5,P2_v_6,P2_v_7,P2_v_8,P2_v_9,P3_n_0,P3_n_1,P3_n_2,P3_n_3,P3_n_4,P3_n_5,P3_n_6,P3_n_7,P3_n_8,P3_n_9,P3_v_0,P3_v_1,P3_v_2,P3_v_3,P3_v_4,P3_v_5,P3_v_6,P3_v_7,P3_v_8,P3_v_9,P4_n_0,P4_n_1,P4_n_2,P4_n_3,P4_n_4,P4_n_5,P4_n_6,P4_n_7,P4_n_8,P4_n_9,P4_v_0,P4_v_1,P4_v_2,P4_v_3,P4_v_4,P4_v_5,P4_v_6,P4_v_7,P4_v_8,P4_v_9,P5_n_0,P5_n_1,P5_n_2,P5_n_3,P5_n_4,P5_n_5,P5_n_6,P5_n_7,P5_n_8,P5_n_9,P5_v_0,P5_v_1,P5_v_2,P5_v_3,P5_v_4,P5_v_5,P5_v_6,P5_v_7,P5_v_8,P5_v_9,P6_n_0,P6_n_1,P6_n_2,P6_n_3,P6_n_4,P6_n_5,P6_n_6,P6_n_7,P6_n_8,P6_n_9,P6_v_0,P6_v_1,P6_v_2,P6_v_3,P6_v_4,P6_v_5,P6_v_6,P6_v_7,P6_v_8,P6_v_9
0,0.045477,0.040022,0.607325,0.037355,0.035648,0.053731,0.038457,0.048750,0.043515,0.049721,0.047188,0.057002,0.045518,0.584409,0.052907,0.044501,0.032437,0.035318,0.054096,0.046623,0.033135,0.030209,0.037054,0.035021,0.039958,0.363147,0.358750,0.028472,0.040540,0.033715,0.030847,0.028576,0.032231,0.030457,0.045489,0.046479,0.033114,0.683844,0.033566,0.035395,0.020093,0.018319,0.219308,0.021240,0.024231,0.277823,0.217501,0.017266,0.163768,0.020451,0.018660,0.017286,0.270623,0.018422,0.027496,0.028211,0.020029,0.557553,0.020303,0.021417,0.052133,0.051371,0.043425,0.060066,0.051297,0.042904,0.043940,0.049191,0.052662,0.553012,0.057534,0.554086,0.053767,0.049154,0.047655,0.045533,0.052089,0.045815,0.045188,0.049180,0.021766,0.025387,0.031008,0.023100,0.272460,0.267962,0.282855,0.023663,0.025925,0.025876,0.027407,0.269519,0.030703,0.025301,0.029695,0.031922,0.271538,0.021230,0.023166,0.269519,0.015573,0.274485,0.015752,0.017298,0.015155,0.240730,0.201527,0.012673,0.016686,0.190122,0.014515,0.507785,0.016995,0.193793,0.015922,0.017781,0.017257,0.017365,0.184314,0.014272
1,0.151717,0.150014,0.052267,0.011673,0.011139,0.016790,0.012017,0.015233,0.426117,0.153033,0.154513,0.379234,0.154007,0.033515,0.015750,0.013247,0.000000,0.210070,0.016130,0.013879,0.016789,0.015305,0.347644,0.017746,0.020243,0.019581,0.017345,0.014427,0.319928,0.210993,0.015581,0.014434,0.016283,0.218397,0.022936,0.518404,0.016724,0.142407,0.016953,0.017881,0.109305,0.000000,0.165219,0.010704,0.012218,0.011810,0.010464,0.350775,0.111582,0.208689,0.113168,0.406255,0.000000,0.208046,0.013818,0.208511,0.010076,0.000000,0.010216,0.010776,0.034949,0.034438,0.029111,0.040321,0.363973,0.028762,0.359084,0.032976,0.035303,0.041083,0.038451,0.038596,0.035952,0.032851,0.031848,0.030430,0.034812,0.362310,0.030199,0.364551,0.012427,0.452119,0.017707,0.013190,0.013269,0.014023,0.013724,0.433960,0.014805,0.014775,0.015770,0.013536,0.017661,0.014559,0.441682,0.442991,0.014698,0.012215,0.013329,0.013558,0.010356,0.011630,0.010476,0.011499,0.010079,0.906516,0.011131,0.000000,0.011098,0.000000,0.388122,0.011090,0.011306,0.387948,0.010595,0.011830,0.011481,0.011553,0.011129,0.144946
2,0.019601,0.017250,0.261728,0.016100,0.015364,0.212832,0.016575,0.400365,0.018755,0.021430,0.019535,0.023597,0.018845,0.239651,0.217250,0.018422,0.013428,0.014621,0.022395,0.412257,0.025016,0.022807,0.027975,0.026440,0.030162,0.029172,0.025844,0.511515,0.030607,0.270463,0.023252,0.513967,0.024293,0.269166,0.034230,0.035035,0.024960,0.023112,0.025305,0.026680,0.682175,0.030206,0.037053,0.035020,0.039949,0.038638,0.034234,0.028472,0.040539,0.033714,0.684118,0.028575,0.032228,0.030456,0.045410,0.046479,0.033115,0.030657,0.033567,0.035395,0.177816,0.551941,0.165729,0.017373,0.014833,0.012405,0.012704,0.014223,0.015226,0.017750,0.016562,0.283543,0.015443,0.014119,0.013695,0.013078,0.014961,0.601494,0.012980,0.014125,0.021766,0.025389,0.031006,0.023107,0.023241,0.024560,0.024037,0.023663,0.025926,0.777304,0.027408,0.269507,0.276697,0.271284,0.029669,0.031932,0.025545,0.021230,0.023165,0.023563,0.000000,0.000000,0.123738,0.000000,0.000000,0.128676,0.000000,0.712346,0.000000,0.000000,0.0

### 2.2. Datos de Encuestas - Números de Empleado

In [29]:
file_path = '../data/Sondeo  -   Perspectiva del Gerente sobre Rotación  (Respuestas).xlsx'

if os.path.exists(file_path):
    datosEncuestas = pd.read_excel(file_path, sheet_name='Respuestas de formulario 1')
    print(f"Archivo cargado exitosamente: {file_path}")
else:
    print(f"ERROR: El archivo no se encuentra en: {file_path}")

Archivo cargado exitosamente: ../data/Sondeo  -   Perspectiva del Gerente sobre Rotación  (Respuestas).xlsx


In [40]:
# Procesar datos de encuestas
datosEncuestas = pd.DataFrame(datosEncuestas).astype('str')
datosEncuestas.columns = datosEncuestas.iloc[0,:]
datosEncuestas.drop([0], inplace=True)
datosEncuestas.reset_index(drop=True, inplace=True)

# Extraer número de empleado
numEmpleado = datosEncuestas[datosEncuestas.columns[1]].to_frame()
numEmpleado.columns = ["num_empleado"]
numEmpleado["num_empleado"] = numEmpleado["num_empleado"].apply(int)

print(f"Total de empleados en encuesta: {len(numEmpleado)}")
numEmpleado.head()

Total de empleados en encuesta: 4995


,num_empleado
0,92109391
1,90391277
2,90269515
3,94566526
4,94296791


In [41]:
df_respuestas = pd.read_excel('../data/Sondeo  -   Perspectiva del Gerente sobre Rotación  (Respuestas).xlsx')
print(df_respuestas.shape)

(4997, 9)


## 3. Procesamiento y Limpieza de Datos

### 3.1. Preparación de Datos Features

In [42]:
df_features = pd.read_csv('../data/df_entrevistados_rotacion.csv')
df_features.head()
print(f"Dimensiones de df_features: {df_features.shape}")

Dimensiones de df_features: (4670, 41)


### 3.2. Integración de Datasets

In [43]:
# Concatenar topics con números de empleado
lda_topics = pd.concat([lda_topics_df, numEmpleado], axis=1)
print(f"Dimensiones del dataset de topics actualizado: {lda_topics.shape}") 


Dimensiones del dataset de topics actualizado: (5033, 121)


In [45]:
# Merge con df_features
df_core = lda_topics.merge(
    df_features, 
    on="num_empleado", 
    how="left"
)
print(f"Dataset integrado - Dimensiones: {df_core.shape}")

Dataset integrado - Dimensiones: (5033, 161)


### 3.4. Eliminación de Columnas con Varianza Cero

In [47]:
# Identificar y eliminar columnas con un solo valor único
dropped_cols = []
for col in df_core.columns:
    if len(df_core[col].unique()) == 1:
        dropped_cols.append(col)

df_core.drop(dropped_cols, axis=1, inplace=True)

print(f"Columnas eliminadas (varianza cero): {len(dropped_cols)}")
if dropped_cols:
    print(f"Columnas eliminadas: {dropped_cols}")
print(f"Dataset final: {df_core.shape}")

Columnas eliminadas (varianza cero): 0
Dataset final: (5033, 161)


### 4.2. Asignación de Nombres de Topics

In [48]:
# Cargar nombres de topics desde archivo pickle
topic_names_path = "../modelo/topic_names.pkl"

if os.path.exists(topic_names_path):
    with open(topic_names_path, "rb") as f:
        topic_names = pickle.load(f)
    
    # Combinar nombres de columnas con nombres de topics
    new_column_names = []
    for idx, (original_col, topic_name) in enumerate(zip(df_core.columns[:120], topic_names)):
        new_column_names.append(f"{original_col}_{topic_name}")
    
    # Agregar columnas restantes
    for col in df_core.columns[120:]:
        new_column_names.append(col)

    df_core.columns = new_column_names
    print(f"Nombres de topics asignados exitosamente")
    print(f"Primeros 5 nombres de topics: {new_column_names[:5]}")
else:
    print(f"ADVERTENCIA: No se encontró el archivo de nombres de topics en {topic_names_path}")

Nombres de topics asignados exitosamente
Primeros 5 nombres de topics: ['P1_n_0_compromiso_y_responsabilidad', 'P1_n_1_beneficios_y_valores_de_la_empresa', 'P1_n_2_prestaciones_y_sueldo', 'P1_n_3_buen_ambiente_laboral', 'P1_n_4_oportunidades_de_desarrollo_y_crecimiento']


## 5. Análisis de Topics

### 5.1. Carga del Diccionario de Topics

In [49]:
topics_dict_path = "../modelo/topics_dict.pkl"

if os.path.exists(topics_dict_path):
    topics_dict = pd.read_pickle(topics_dict_path)
    print(f"Diccionario de topics cargado exitosamente")
    print(f"Preguntas analizadas: {list(topics_dict.keys())}")
else:
    print(f"ADVERTENCIA: No se encontró el archivo de diccionario de topics en {topics_dict_path}")

Diccionario de topics cargado exitosamente
Preguntas analizadas: ['P1_n', 'P1_v', 'P2_n', 'P2_v', 'P3_n', 'P3_v', 'P4_n', 'P4_v', 'P5_n', 'P5_v', 'P6_n', 'P6_v']


### 5.2. Función para Visualizar Topics

In [50]:
def print_topics(topics):
    """
    Imprime los topics de manera formateada.
    
    Parámetros:
    -----------
    topics : list
        Lista de tuplas (topic_id, topic_words)
    """
    for topic_id, topic_words in topics:
        print(f"Topic #{topic_id}: {topic_words}")
    return None

### 5.3. Visualización de Topics por Pregunta

In [51]:
if 'topics_dict' in locals():
    for col in topics_dict:
        print("="*80)
        print(f"Pregunta: {col}")
        print("="*80)
        print_topics(topics_dict[col].tolist())
        print("\n")
else:
    print("No se puede visualizar topics: diccionario no cargado")

Pregunta: P1_n
Topic #0: 0.167*"compromiso" + 0.098*"responsabilidad" + 0.082*"empresa" + 0.044*"cambios" + 0.042*"comodidad" + 0.041*"resultados" + 0.039*"tener" + 0.035*"cambio" + 0.034*"procesos" + 0.032*"actividades"
Topic #1: 0.329*"empresa" + 0.242*"beneficios" + 0.094*"valores" + 0.077*"colaboradores" + 0.036*"otorga" + 0.031*"bien" + 0.028*"hacer" + 0.022*"encuentran" + 0.020*"pertenencia" + 0.019*"gran"
Topic #2: 0.716*"prestaciones" + 0.093*"empresa" + 0.050*"sueldo" + 0.050*"buenas" + 0.018*"utilidades" + 0.013*"incentivos" + 0.010*"buenos" + 0.008*"jefes" + 0.007*"excelentes" + 0.006*"cuenta"
Topic #3: 0.379*"laboral" + 0.316*"ambiente" + 0.124*"buen" + 0.046*"prestaciones" + 0.017*"ingreso" + 0.016*"facilidad" + 0.016*"vida" + 0.012*"manera" + 0.012*"calidad" + 0.011*"forma"
Topic #4: 0.141*"ofrece" + 0.131*"empresa" + 0.116*"desarrollo" + 0.074*"dentro" + 0.052*"crecimiento" + 0.044*"coppel" + 0.030*"cultura" + 0.028*"misma" + 0.028*"oportunidad" + 0.024*"seguir"
Topic #5

## 7. Exploración del Dataset Final

In [52]:
print("Información del Dataset Final:")
print("="*80)
print(f"Dimensiones: {df_core.shape}")
print(f"\nColumnas totales: {len(df_core.columns)}")
print(f"\nPrimeras columnas (Topics):")
print(list(df_core.columns[:10]))
print(f"\nÚltimas columnas (Variables organizacionales):")
print(list(df_core.columns[-10:]))

Información del Dataset Final:
Dimensiones: (5033, 161)

Columnas totales: 161

Primeras columnas (Topics):
['P1_n_0_compromiso_y_responsabilidad', 'P1_n_1_beneficios_y_valores_de_la_empresa', 'P1_n_2_prestaciones_y_sueldo', 'P1_n_3_buen_ambiente_laboral', 'P1_n_4_oportunidades_de_desarrollo_y_crecimiento', 'P1_n_5_satisfaccion_y_gusto_por_el_trabajo', 'P1_n_6_crecimiento_y_estabilidad_laboral', 'P1_n_7_estabilidad_y_seguridad_laboral', 'P1_n_8_identificacion_con_la_empresa_y_balance', 'P1_n_9_buen_trato_y_apoyo']

Últimas columnas (Variables organizacionales):
['promedio_grupo_salarial', 'std_grupo_salarial', 'moda_grupo_salarial', 'promedio_antiguedad_subordinados_anos', 'std_antiguedad_equipo', 'antiguedad_anois', 'tamano_equipo', 'pct_ocupacion_equipo', 'indice_rigidez_contratacion', 'pct_posiciones_vacantes']


In [54]:
df_core.head()

,P1_n_0_compromiso_y_responsabilidad,P1_n_1_beneficios_y_valores_de_la_empresa,P1_n_2_prestaciones_y_sueldo,P1_n_3_buen_ambiente_laboral,P1_n_4_oportunidades_de_desarrollo_y_crecimiento,P1_n_5_satisfaccion_y_gusto_por_el_trabajo,P1_n_6_crecimiento_y_estabilidad_laboral,P1_n_7_estabilidad_y_seguridad_laboral,P1_n_8_identificacion_con_la_empresa_y_balance,P1_n_9_buen_trato_y_apoyo,P1_v_0_compromiso_y_lealtad,P1_v_1_oferta_de_prestaciones_de_la_empresa,P1_v_2_necesidad_de_un_trabajo_estable,P1_v_3_compensacion_y_condiciones_laborales,P1_v_4_satisfaccion_con_el_trabajo,P1_v_5_ambiente_y_desarrollo_laboral,P1_v_6_apoyo_de_la_empresa_a_colaboradores,P1_v_7_comodidad_y_beneficios_adicionales,P1_v_8_oportunidades_de_crecimiento_y_beneficios,P1_v_9_estabilidad_y_seguridad_laboral,P2_n_0_busqueda_de_mejores_condiciones_laborales,P2_n_1_desadaptacion_al_ritmo_de_trabajo,P2_n_2_problemas_con_horarios_y_jefes,P2_n_3_insatisfaccion_con_las_actividades,P2_n_4_exceso_de_carga_de_trabajo_y_exigencia,P2_n_5_falta_de_compromiso_y_adaptacion,P2_n_6_sueldo_bajo_y_horarios_demandantes,P2_n_7_expectativas_incumplidas_sobre_pago_y_jornada,P2_n_8_mal_trato_y_falta_de_seguimiento_de_lideres,P2_n_9_falta_de_adaptacion_al_ritmo_de_la_empresa,P2_v_0_busqueda_de_mejores_oportunidades_y_ambiente,P2_v_1_mal_trato_y_baja_compensacion,P2_v_2_insatisfaccion_con_tareas_asignadas,P2_v_3_falta_de_adaptacion_y_gusto_por_el_trabajo,P2_v_4_exceso_de_carga_y_ritmo_de_trabajo,P2_v_5_falta_de_liderazgo_y_seguimiento,P2_v_6_mejores_ofertas_salariales_y_prestaciones,P2_v_7_desmotivacion_por_metas_y_falta_de_adaptacion,P2_v_8_horarios_extensos_y_expectativas_no_cumplidas,P2_v_9_conflicto_entre_horarios_y_responsabilidades_personales,P3_n_0_busqueda_de_mejores_condiciones_laborales,P3_n_1_desgaste_y_cambio_generacional,P3_n_2_conflictos_con_jefes_y_horarios,P3_n_3_insatisfaccion_con_las_tareas_y_responsabilidades,P3_n_4_exceso_de_carga_de_trabajo_y_presion,P3_n_5_perdida_de_compromiso_y_responsabilidad,P3_n_6_sueldo_poco_competitivo_y_malos_horarios,P3_n_7_expectativas_economicas_no_cumplidas,P3_n_8_mal_trato_por_parte_de_lideres,P3_n_9_falta_de_adaptacion_a_cambios_en_el_trabajo,P3_v_0_busqueda_de_mejores_oportunidades_laborales,P3_v_1_mejor_oferta_salarial_y_de_trato,P3_v_2_rutina_y_falta_de_nuevos_retos,P3_v_3_perdida_de_interes_y_desadaptacion,P3_v_4_exceso_de_carga_y_ritmo_de_trabajo,P3_v_5_mal_liderazgo_y_falta_de_compromiso,P3_v_6_ofertas_con_mejor_sueldo_y_prestaciones,P3_v_7_desgaste_y_desmotivacion_por_metas,P3_v_8_horarios_extensos_y_desgaste_laboral,P3_v_9_conflicto_entre_horarios_y_vida_personal,P4_n_0_alta_competencia_laboral_en_la_zona,P4_n_1_cultura_de_alta_exigencia_y_metas,P4_n_2_caracteristicas_especificas_de_la_zona,P4_n_3_poca_afluencia_de_clientes_e_inseguridad,P4_n_4_falta_de_personal_comprometido_por_mejores_ofertas,P4_n_5_ubicacion_y_formato_de_la_tienda,P4_n_6_inseguridad_y_riesgo_de_robo,P4_n_7_reto_de_liderar_colaboradores,P4_n_8_falta_de_personal_y_cambios_organizacionales,P4_n_9_entorno_competitivo_y_carga_de_trabajo,P4_v_0_competencia_por_talento_con_mejores_condiciones,P4_v_1_condiciones_de_inseguridad_del_entorno,P4_v_2_poca_afluencia_de_clientes_y_carga_de_trabajo,P4_v_3_tipo_de_clientela_y_ubicacion_de_la_tienda,P4_v_4_cultura_orientada_a_metas_de_venta,P4_v_5_falta_de_personal_y_problemas_de_seguridad,P4_v_6_problemas_de_seguridad_y_transporte,P4_v_7_alta_oferta_laboral_en_la_zona,P4_v_8_ambiente_laboral_y_demanda_de_clientes,P4_v_9_gestion_de_colaboradores_y_cambio,P5_n_0_mejores_salarios_e_incentivos,P5_n_1_gestion_de_personal_y_carga_de_trabajo,P5_n_2_mejores_herramientas_y_horarios,P5_n_3_trabajo_en_equipo_y_apoyo,P5_n_4_mejora_del_ambiente_laboral_y_liderazgo,P5_n_5_aumento_de_sueldo_y_apoyo_de_transporte,P5_n_6_mejores_sueldos_y_horarios_flexibles,P5_n_7_planes_de_desarrollo_y_seguimiento,P5_n_8_mejora_del_esquema_de_incentivos_y_metas,P5_n_9_mejores_prestaciones_y_beneficios,P5_v_0_esquemas_de_incentivos_y_metas_alcanzables,P5_v_1_

In [56]:
df_core.to_csv('../data/df_core_final.csv', index=False)